# HoloKai: frozen ModernBERT NER baselines — validation only

Reuse the existing selected model artifacts and saved validation metrics for both encoders. Reporting reads saved results without recomputing predictions. This revision has not been executed.

The preserved training settings are identical linear BIO heads, head LR `1e-3`, seed 42, batch size 8, 10 epochs, AdamW with zero weight decay, and gradient clipping 1.0. Both pretrained encoders remain frozen; no hyperparameter search or full fine-tuning. Checkpoint selection uses exact-span validation micro F1, retaining the earliest best epoch.

Only Train and Validation from `week8/Holokai_Option_B_Annotation_Files` are used. Preserve the original data and existing 11-label schema. Option B additions remain automated and unverified.

The two scripts remain in `py/`. Existing verified ONNX artifacts are reused; ONNX export and inference verification are unchanged. The final report contains one overall comparison table and all 11 per-label metrics for each model.


## 1. Environment setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.53.3", "onnx==1.18.0", "onnxruntime==1.22.1"], check=True)
print("DEPENDENCIES_INSTALLED")

## 2. Mount Google Drive and verify the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "No GPU connected. Select Runtime > Change runtime type > L4 GPU (or T4)."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__)

## 3. Paths, read-only data checks, and shared execution settings

In [ ]:
from pathlib import Path
import hashlib, json, os
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
PROJECT = Path("/content/drive/MyDrive/Holokai_Project")
ROOT = PROJECT / "week8/modernbert_baseline"
DATA = PROJECT / "week8/Holokai_Option_B_Annotation_Files"
PY = ROOT / "py"
OUT = ROOT / "output"
SCRATCH = Path("/content/modernbert_baseline_scratch")
assert ROOT.is_dir() and DATA.is_dir(), "Expected Drive project folders are missing"
assert {p.name for p in ROOT.iterdir() if p.is_dir()} == {"py", "notebook", "output"}
for script in ("train_eval.py", "export_onnx.py"):
    assert (PY / script).is_file(), f"Missing {script}"
OUT.mkdir(exist_ok=True)
SCRATCH.mkdir(exist_ok=True)
sys.path.insert(0, str(PY))
from train_eval import load_split, MODELS, LABELS, TAGS, decode, score, word_labels
SOURCE_HASHES = {}
for split in ("train", "validation"):
    rows, manifest = load_split(DATA, split)
    SOURCE_HASHES[split] = manifest
    for row in rows:
        assert sorted(decode(word_labels(row).tolist())) == sorted(row["ner"]), "BIO roundtrip changed an annotation"
    perfect = score(rows, [r["ner"] for r in rows])
    assert perfect["overall"]["f1"] == 1.0
    print(split, manifest["records"], "records; SHA256", manifest["sha256"])
assert len(LABELS) == 11 and len(TAGS) == 23
COMMON = ["--data", str(DATA)]

def command(script, args):
    env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1")
    argv = [sys.executable, "-u", str(PY / script), *args]
    with subprocess.Popen(argv, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, argv)

def train_model(name):
    destination = OUT / name
    metrics_file = destination / "metrics.json"
    if metrics_file.exists():
        metrics = json.loads(metrics_file.read_text())
        assert metrics["model_id"] == MODELS[name]
        assert {split: metrics["data"][split] for split in SOURCE_HASHES} == SOURCE_HASHES, "Existing results used different data"
        assert metrics["hyperparameters"]["lr"] == 1e-3 and metrics["hyperparameters"]["seed"] == 42
        assert metrics["hyperparameters"]["epochs"] == 10 and metrics["hyperparameters"]["batch_size"] == 8
        if not metrics["onnx"]["verified"]:
            assert (SCRATCH / name / "best_head.pt").is_file(), "Training exists but temporary head is unavailable; do not overwrite results"
        print("Using completed training:", name)
        return
    command("train_eval.py", COMMON + ["--model", name, "--output", str(destination),
            "--scratch", str(SCRATCH / name), "--lr", "0.001", "--seed", "42", "--epochs", "10", "--batch-size", "8"])

def export_model(name):
    metrics = json.loads((OUT / name / "metrics.json").read_text())
    if metrics["onnx"]["verified"]:
        print("Verified export already exists:", name)
        return
    command("export_onnx.py", COMMON + ["--output", str(OUT / name), "--scratch", str(SCRATCH / name)])
print("PREFLIGHT_PASSED")

## 4. Train and evaluate ModernBERT-large

In [ ]:
train_model("modernbert_large")

## 5. Export ModernBERT-large and verify ONNX inference

In [ ]:
export_model("modernbert_large")


## 6. Train and evaluate BioClinical-ModernBERT-large

In [ ]:
train_model("bioclinical_modernbert_large")

## 7. Export BioClinical-ModernBERT-large and verify ONNX inference

In [ ]:
export_model("bioclinical_modernbert_large")

## 8. Validation comparison and all 11 per-label metrics

Read the existing saved validation results from each selected checkpoint. No predictions or metrics are recomputed in this reporting cell.


In [ ]:
import html
from IPython.display import display, HTML
METRICS = ["precision", "recall", "f1", "tp", "fp", "fn"]

def show_table(title, rows, columns):
    def value(row, column):
        item = row[column]
        return f"{item:.4f}" if column in ("precision", "recall", "f1") else str(item)
    rendered = "<h3>" + html.escape(title) + "</h3><table><tr>"
    rendered += "".join("<th>" + html.escape(c) + "</th>" for c in columns) + "</tr>"
    for row in rows:
        rendered += "<tr>" + "".join("<td>" + html.escape(value(row, c)) + "</td>" for c in columns) + "</tr>"
    display(HTML(rendered + "</table>"))

completed, overall_rows, per_label_rows = [], [], []
for name in MODELS:
    directory = OUT / name
    metrics = json.loads((directory / "metrics.json").read_text())
    assert metrics["model_id"] == MODELS[name]
    assert metrics["onnx"]["verified"] and metrics["encoder_frozen_verified"]
    for artifact in ("metrics.json", "per_label_metrics.csv", "predictions.json", "model.onnx", "tokenizer.json", "inference_config.json"):
        assert (directory / artifact).is_file() and (directory / artifact).stat().st_size > 0, artifact
    with (directory / "model.onnx").open("rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == metrics["onnx"]["sha256"]
    predictions = json.loads((directory / "predictions.json").read_text())["validation"]
    assert len(predictions) == SOURCE_HASHES["validation"]["records"]
    validation = metrics["validation"]
    assert set(validation["per_label"]) == set(LABELS) and len(LABELS) == 11
    overall_rows.append({"model": name, **validation["overall"]})
    per_label_rows.extend({"model": name, "label": label, **validation["per_label"][label]} for label in LABELS)
    completed.append(metrics)
assert completed[0]["hyperparameters"] == completed[1]["hyperparameters"]
assert completed[0]["head_initial_weight_sha256"] == completed[1]["head_initial_weight_sha256"]
for split in SOURCE_HASHES:
    _, current = load_split(DATA, split)
    assert current == SOURCE_HASHES[split], "Source data changed"
show_table("Validation overall (micro)", overall_rows, ["model", *METRICS])
show_table("Validation per label (all 11 labels per model)", per_label_rows, ["model", "label", *METRICS])
print("Validation report uses existing selected models and saved metrics; no recomputation.")


## 9. Frozen NER-head learning-rate experiment — code ready, not executed

Test **3e-4, 1e-3, 3e-3** separately for each of the two original encoders. Both encoders stay frozen in evaluation mode. Reuse the existing **1024 → 23 linear BIO head architecture**, resetting the same seed-42 initialization and training order for every LR; no warm starts between trials. Frozen features are computed once per model. Each trial runs **at most 20 epochs**, with **early-stopping patience 3** (strict F1 improvement; min delta 0). Select the best checkpoint and LR by **Validation strict entity-level micro F1**: exact inclusive word start, end, and label. Ties retain the earliest best epoch and first LR in the listed order.

Use only the Week 8 Option B **Train (1,509)** and **Validation (171)** records, preserving the 11-label schema and original data. All other settings remain unchanged: seed 42, batch size 8, AdamW, weight decay 0, gradient clipping 1.0, FP32, existing tokenizer/alignment/decoder/scorer. Encoder revisions are pinned to the original baseline metrics.

**Execution order when authorized:** run the three new cells below (setup → both model sweeps → best-run ONNX verification and reporting). They are self-contained; the original baseline and OpenMed cells do not need to run. `RUN_HEAD_SWEEP = False` keeps this section disabled until execution is explicitly enabled. **No cells in this section have been run.**

New outputs go only to `output/head_lr_sweep/{modernbert_large,bioclinical_modernbert_large}/`. Each model keeps one `metrics.json` (three compact trial summaries plus winning history and full metrics), one `per_label_metrics.csv`, one `predictions.json`, and only the winner's ONNX/tokenizer/inference artifacts. A small selected `best_head.pt` is kept until the existing ONNX exporter verifies and removes it. No per-LR model copies or plots. Existing baseline and OpenMed results remain unchanged. Reporting displays one overall comparison table plus all 11 per-label P/R/F1/TP/FP/FN for each winning run.


In [ ]:
# Code only: leave False until the experiment is explicitly authorized to run.
RUN_HEAD_SWEEP = True
if RUN_HEAD_SWEEP:
    import subprocess, sys, os, json
    from pathlib import Path
    sys.dont_write_bytecode = True
    os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.53.3', 'onnx==1.18.0', 'onnxruntime==1.22.1'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    import torch
    assert torch.cuda.is_available(), 'Select an L4 GPU runtime (T4 also supported).'
    SWEEP_ROOT = Path('/content/drive/MyDrive/Holokai_Project/week8/modernbert_baseline')
    SWEEP_DATA = SWEEP_ROOT.parent / 'Holokai_Option_B_Annotation_Files'
    SWEEP_PY = SWEEP_ROOT / 'py'
    SWEEP_OUT = SWEEP_ROOT / 'output/head_lr_sweep'
    SWEEP_MODELS = {'modernbert_large': 'answerdotai/ModernBERT-large',
                    'bioclinical_modernbert_large': 'thomas-sounack/BioClinical-ModernBERT-large'}
    assert SWEEP_DATA.is_dir()
    for script in ('train_eval.py', 'export_onnx.py'):
        assert (SWEEP_PY / script).is_file()
    def head_sweep_command(script, args):
        env = dict(os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false',
                   HF_HUB_DISABLE_TELEMETRY='1')
        argv = [sys.executable, '-u', str(SWEEP_PY / script), *args]
        with subprocess.Popen(argv, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
            status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, argv)
    print('Head sweep ready:', torch.cuda.get_device_name(0), '; outputs:', SWEEP_OUT)
else:
    print('Head LR experiment disabled; no setup, training, evaluation, or export started.')


In [ ]:
# Run only this new section when authorized; original baseline/OpenMed cells are unnecessary.
if RUN_HEAD_SWEEP:
    for name, model_id in SWEEP_MODELS.items():
        baseline = json.loads((SWEEP_ROOT / 'output' / name / 'metrics.json').read_text())
        assert baseline['model_id'] == model_id and baseline['encoder_frozen_verified']
        for key, value in {'seed': 42, 'batch_size': 8, 'optimizer': 'AdamW',
                           'weight_decay': 0.0, 'max_grad_norm': 1.0, 'precision': 'float32'}.items():
            assert baseline['hyperparameters'][key] == value, f'Original {key} differs'
        assert baseline['revision'], 'The original encoder revision must be pinned'
        destination = SWEEP_OUT / name
        # The one selected head stays in Drive until ONNX export verifies and removes it.
        head_sweep_command('train_eval.py', [
            '--model', name, '--data', str(SWEEP_DATA), '--output', str(destination),
            '--scratch', str(destination), '--lr-sweep', '--seed', '42', '--batch-size', '8',
            '--revision', baseline['revision']])
        result = json.loads((destination / 'metrics.json').read_text())
        assert result['head_initial_weight_sha256'] == baseline['head_initial_weight_sha256']
        assert result['data'] == {split: baseline['data'][split] for split in ('train', 'validation')}
        assert result['encoder_frozen_verified'] and len(result['sweep']['trials']) == 3
        assert result['sweep']['max_epochs'] == 20 and result['sweep']['patience'] == 3
    print('HEAD_LR_SWEEP_COMPLETE: both frozen encoders; Train + Validation only.')


In [ ]:
# Export only each model's winning LR/checkpoint; preserve all original artifacts.
if RUN_HEAD_SWEEP:
    import html
    from IPython.display import display, HTML
    sweep_results = {}
    for name in SWEEP_MODELS:
        destination = SWEEP_OUT / name
        metrics_path = destination / 'metrics.json'
        result = json.loads(metrics_path.read_text())
        if not result['onnx']['verified']:
            head_sweep_command('export_onnx.py', [
                '--data', str(SWEEP_DATA), '--output', str(destination),
                '--scratch', str(destination)])
        result = json.loads(metrics_path.read_text())
        assert result['onnx']['verified'] and (destination / 'model.onnx').is_file()
        assert len(result['validation']['per_label']) == 11
        sweep_results[name] = result
    assert len({r['head_initial_weight_sha256'] for r in sweep_results.values()}) == 1
    def head_sweep_table(title, rows, columns):
        def cell(value):
            return html.escape(f'{value:.4f}' if isinstance(value, float) else str(value))
        body = ''.join('<tr>' + ''.join(f'<td>{cell(row[c])}</td>' for c in columns) + '</tr>' for row in rows)
        display(HTML('<h3>' + html.escape(title) + '</h3><table><tr>' +
                     ''.join('<th>' + html.escape(c) + '</th>' for c in columns) + '</tr>' + body + '</table>'))
    metric_columns = ['precision', 'recall', 'f1', 'tp', 'fp', 'fn']
    head_sweep_table('Best LR per model — Validation strict entity-level micro metrics', [
        dict(model=name, lr=r['sweep']['selected_lr'], epoch=r['selected_epoch'],
             **r['validation']['overall']) for name, r in sweep_results.items()],
        ['model', 'lr', 'epoch', *metric_columns])
    head_sweep_table('Best runs — all 11 Validation labels', [
        dict(model=name, label=label, **values)
        for name, r in sweep_results.items() for label, values in r['validation']['per_label'].items()],
        ['model', 'label', *metric_columns])
    print('Best-run metrics, predictions, and verified ONNX saved in', SWEEP_OUT)
